# Experiment 47 — Allocate-on-arrival SparseWalker

Test the idea that a sparse walk should **write the label where it arrives** instead of learning to find a globally pre-assigned label address.

Three strict forward-only arms:
- `allocate_only`: bind target to an empty reached concept
- `allocate_path`: also reinforce only the touched 2-hop path
- `allocate_rewire`: also create a local shortcut toward an existing target alias when the reached region cannot allocate

Each concept owns one label; each item may have up to four aliases. Inference reads only the K active terminal concepts. No optimizer, no `backward()`, no autograd gradients.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, torch
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='research/active'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
for p in [f'{REPO}/src',f'{REPO}/experiments']:
    if p not in sys.path: sys.path.insert(0,p)
HEAD=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,flush=True)
print('TORCH',torch.__version__,flush=True)
print('BRANCH',BRANCH,flush=True)
print('HEAD',HEAD,flush=True)
import sparsewalker
print('IMPORT_OK',sparsewalker.__file__,flush=True)
assert torch.cuda.is_available(), 'GPU runtime required'


## Run

The first epoch mostly creates memory. Watch `val_target_in_sparse_candidates` as well as NDCG: if candidate recall rises but NDCG stays weak, routing/ranking is the problem; if candidate recall itself stays low, allocation/locality is the problem.


In [ ]:
import runpy, sys, os
EPOCHS=8
SCRIPT=f'{REPO}/experiments/run_ml1m_allocate_on_arrival.py'
text=Path(SCRIPT).read_text()
assert 'Experiment 47' in text and 'allocate_on_arrival' in SCRIPT, 'stale script clone'
argv=[SCRIPT,
      '--epochs-per-arm',str(EPOCHS),
      '--batch-size','512',
      '--eval-batch-size','1024',
      '--progress-every','0',
      '--data-dir','/content/drive/MyDrive/sparsewalker_data']
print('RUNNING_IN_PROCESS',' '.join(argv),flush=True)
old_argv=sys.argv[:]; old_cwd=os.getcwd()
sys.argv=argv; os.chdir(REPO)
try:
    runpy.run_path(SCRIPT,run_name='__main__')
finally:
    sys.argv=old_argv; os.chdir(old_cwd)


## Ranked result


In [ ]:
import pandas as pd, json
root=Path('/content/drive/MyDrive/sparsewalker_allocate_on_arrival/seed42')
sp=root/'summary.json'
if sp.exists():
    summary=json.loads(sp.read_text())
    display(pd.DataFrame(summary['ranking']))
    print(json.dumps(summary,indent=2))
else:
    print('No summary yet.')


## Per-arm trajectories


In [ ]:
for arm in ['allocate_only','allocate_path','allocate_rewire']:
    hp=root/f'history_{arm}.json'
    if not hp.exists(): continue
    h=pd.DataFrame(json.loads(hp.read_text()))
    cols=['epoch','val_NDCG@10','val_HR@10','val_target_in_sparse_candidates','val_mean_positive_candidates','train_memory_hit_rate','allocations','rewires','occupied_concepts','item_coverage','positions_per_s']
    print('\n',arm)
    display(h[[c for c in cols if c in h.columns]])
